In [1]:
import sys
import os

# Get the absolute path to the project directory
project_dir = os.path.abspath("..")

# Append the project directory to sys.path
if project_dir not in sys.path:
    sys.path.append(project_dir)
    
from src.predictionModule.LoadupSamples import LoadupSamples
from src.predictionModule.FilterSamples import FilterSamples

import numpy as np
import pandas as pd
import polars as pl
import datetime
import scipy
import matplotlib.pyplot as plt
import optuna
import random

from itertools import product
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

import logging
formatted_date = datetime.datetime.now().strftime("%d%b%y_%H%M").lower()

logger = logging.getLogger()
logger.setLevel(logging.DEBUG)
handler = logging.StreamHandler(sys.stdout)
formatter = logging.Formatter(fmt="%(asctime)s - %(message)s")
handler.setFormatter(formatter)
if not logger.hasHandlers():
    logger.addHandler(handler)
else:
    logger.handlers[:] = [handler]

#Output File handler
formatted_str = f"notebook-stomp-{formatted_date}"
file_handler = logging.FileHandler(f"{formatted_str}.log", mode="w")
file_handler.setFormatter(formatter)
logger.addHandler(file_handler)

# Usage
logger.setLevel(logging.INFO)
logger.info("This will print to the notebook's output cell")

2025-09-09 11:48:04,208 - This will print to the notebook's output cell


c:\Users\kimer\Desktop\RandomOdyssey\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
params = {
    "idxAfterPrediction": 5,
    'timesteps': 90,
    'target_option': 'last',
    "LoadupSamples_time_scaling_stretch": True,
    "LoadupSamples_time_inc_factor": 61,

    "FilterSamples_q_up": 0.985,
    
    "FilterSamples_cat_over20": True,
    "FilterSamples_cat_posOneYearReturn": False,
    "FilterSamples_cat_posFiveYearReturn": False,
}

In [ ]:
timegroup = "group_regOHLCV_over5years"
treegroup = "group_debug"

eval_date = datetime.date(year=2025, month=7, day=13)
evaldates = [eval_date - datetime.timedelta(days=i) for i in range(1, 6)]
start_train_date = datetime.date(year=2014, month=1, day=1)
split_Date = datetime.date(year=2025, month=1, day=1)
ls = LoadupSamples(
    train_start_date=start_train_date,
    test_dates=evaldates,
    treegroup=treegroup,
    timegroup=timegroup,
    params=params,
)
ls.load_samples(main_path = "../src/featureAlchemy/bin/")
ls.split_dataset(
    start_date=start_train_date,
    last_train_date=split_Date,
    last_test_date=eval_date
)
fs_pre = FilterSamples(
    Xtree_train = ls.train_Xtree, 
    ytree_train = ls.train_ytree, 
    treenames   = ls.featureTreeNames,
    Xtree_test  = ls.test_Xtree,  
    ytree_test  = ls.test_ytree,
    meta_train  = ls.meta_pl_train, 
    meta_test   = ls.meta_pl_test, 
    params      = params
)
mask_train_pre, mask_test_pre = fs_pre.categorical_masks()
ls.apply_masks(mask_train_pre, mask_test_pre)

In [4]:
Xtree_train = ls.train_Xtree
ytree_train = ls.train_ytree
Xtree_test  = ls.test_Xtree
ytree_test  = ls.test_ytree

Xtime_train = ls.train_Xtime
ytime_train = ls.train_ytime
Xtime_test  = ls.test_Xtime
ytime_test  = ls.test_ytime

treenames   = ls.featureTreeNames
timenames   = ls.featureTimeNames
meta_train  = ls.meta_pl_train
meta_test   = ls.meta_pl_test

dates_tr = meta_train['date'].unique().sort()
dates_te = meta_test['date'].unique().sort()

from src.common.DataFrameTimeOperations import DataFrameTimeOperations as dfta
dates_tr_idx = dfta(meta_train, 'date').getNextLowerOrEqualIndices(dates_tr)
dates_te_idx = dfta(meta_test, 'date').getNextLowerOrEqualIndices(dates_te)

assert not any([i == -1 for i in dates_tr_idx])
assert not any([i == -1 for i in dates_te_idx])

In [ ]:
# ---- knobs (use existing globals if present) ----
BASE_RS = 42
KM_INIT = "k-means++"

n_splits = 200
n_test_days = 15

assert "Xtime_train" in globals() and "ytree_train" in globals(), "Need Xtime_train/ytree_train"
nS, nT, nF = Xtime_train.shape
ytr_arr = np.asarray(ytree_train)
ytr_vec = ytr_arr[:, -1] if ytr_arr.ndim == 2 else ytr_arr

yte_arr = np.asarray(ytree_test)
yte_vec = yte_arr[:, -1] if yte_arr.ndim == 2 else yte_arr

In [6]:
def geometric_mean_safe(arr):
    arr = np.asarray(arr, dtype=float)
    minv = np.min(arr) if arr.size else 0.0
    shift = -minv + 1e-9 if minv <= 0 else 0.0
    return float(np.exp(np.mean(np.log(arr + shift)))) if arr.size else np.nan

def metric(arr):
    """Custom cluster score function."""
    gm = geometric_mean_safe(arr)
    return gm - 1

def make_design(X, t_win, feat_idx):
    feat_idx = np.atleast_1d(feat_idx)
    Xw = X[:, -t_win:, feat_idx]
    return Xw.reshape(Xw.shape[0], -1)

def _score_once(t_win:int, k:int, f_idcs: list[int], s_tr_l:int, s_tr_u:int, s_te_l:int, s_te_u:int, do_transform: bool = False) -> float:
    """Return paired test score for the train-best cluster (one run)."""

    Xtr, ytr = Xtime_train[s_tr_l:s_tr_u], ytr_vec[s_tr_l:s_tr_u]
    Xte, yte = Xtime_train[s_te_l:s_te_u], ytr_vec[s_te_l:s_te_u]

    if k >= Xtr.shape[0]:
        return -np.inf

    # design matrices
    Xd_tr = make_design(Xtr, t_win, f_idcs)
    Xd_te = make_design(Xte, t_win, f_idcs)

    if do_transform:
        scaler = StandardScaler().fit(Xd_tr)
        Xd_tr = scaler.transform(Xd_tr)
        Xd_te = scaler.transform(Xd_te)

    km = KMeans(n_clusters=k, n_init=k, init=KM_INIT).fit(Xd_tr)
    lab_tr = km.labels_
    lab_te = km.predict(Xd_te)

    # cluster-wise scores
    tr_scores, te_scores = [], []
    for c in range(k):
        idx_tr = np.where(lab_tr == c)[0]
        tr_scores.append(metric(ytr[idx_tr]) if idx_tr.size else np.nan)
        idx_te = np.where(lab_te == c)[0]
        te_scores.append(metric(yte[idx_te]) if idx_te.size else np.nan)

    if not np.any(np.isfinite(tr_scores)):
        return -np.inf
    best_c = int(np.nanargmax(tr_scores))
    paired = te_scores[best_c]
    logger.info(f"t_win={t_win}, k={k}, tr_size={Xtr.shape[0]}, te_size={Xte.shape[0]} -> best_c={best_c}, paired={paired}")
    return float(paired) if np.isfinite(paired) else -np.inf

In [ ]:
max_training_days = 1200
N = len(dates_tr_idx)
lo = max_training_days - 1                      # min pivot (last train index)
hi = N - n_test_days - 1                      # max pivot
eligible = list(range(lo, hi + 1))
assert len(eligible) >= n_splits, f"Too few eligible pivots ({len(eligible)}) for n_splits={n_splits}"
pivots = sorted(random.sample(eligible, n_splits))

tr_slice_list = [slice(p - max_training_days + 1, p + 1) for p in pivots]      # [..)
te_slice_list = [slice(p + 1, p + 1 + n_test_days) for p in pivots]          # [..)

for p in pivots:
    logger.info(f"  Pivot {p}: Date {dates_tr[p]}")

In [ ]:
def make_objective():
    def objective(trial: optuna.Trial) -> float:
        # search space
        t_win = trial.suggest_int("TIME_WINDOW", 25, 41, step = 2)
        k = trial.suggest_int("N_CLUSTERS", 18, 30, step = 2)
        n_training_days = trial.suggest_int("N_TRAIN_DAYS", 800, 1150, step = 50)
        do_transform = False
        f_idcs_cat = 2

        if f_idcs_cat == 0:       f_idcs = [0]
        elif f_idcs_cat == 1:     f_idcs = [1]
        elif f_idcs_cat == 2:     f_idcs = [0, 1]

        scores = []
        for i in range(n_splits):
            tr_idx = tr_slice_list[i]
            te_idx = te_slice_list[i]

            # example: get date bounds if needed
            s_tr_l, s_tr_u = dates_tr_idx[tr_idx.stop - n_training_days + 1], dates_tr_idx[tr_idx.stop] - 1
            s_te_l, s_te_u = dates_tr_idx[te_idx.start], dates_tr_idx[te_idx.stop] - 1

            try:
                sc = _score_once(t_win, k, f_idcs, s_tr_l, s_tr_u, s_te_l, s_te_u, do_transform=do_transform)
            except Exception as e:
                logger.info(f"Exception during scoring: {e}")
                sc = -np.inf
            scores.append(sc)

        vals = [v for v in scores if np.isfinite(v)]
        vals = np.array(vals)
        vals_log = np.log(1.0 + vals)
        if len(vals) < (len(scores)//2):
            return 0.0
        return float(np.mean(vals_log)) if len(vals_log) else -np.inf
    return objective

In [ ]:
studytime = 60*60*6
n_startup_trials = 50
studyname = f"optuna_clustering_idea_{formatted_str}"
device = "cuda"

In [ ]:
# === Optuna driver ===
optuna.logging.enable_propagation()
sampler = optuna.samplers.TPESampler(n_startup_trials=n_startup_trials)
study = optuna.create_study(
    study_name=studyname,
    storage="sqlite:///sandbox_optuna.db",
    direction="maximize",
    load_if_exists=True,
    sampler=sampler,
)
study.optimize(make_objective(), timeout=studytime)

logger.info(f"Best parameters: {study.best_params}")
logger.info(f"Best score: {study.best_value}")

df: pd.DataFrame = study.trials_dataframe()
logger.info("\nTrials DataFrame:")
logger.info(df.sort_values("value").to_string())

param_importances = optuna.importance.get_param_importances(study)
logger.info("Parameter Importances:")
for key, value in param_importances.items():
    logger.info(f"{key}: {value}")